# 05 - Features, split and first models

Goal: fix the features, split the incidents by date, build one preprocessing pipeline, and compare a baseline and three models on the validation set. The test set is not used here.

## 1. Loading

In [6]:
import sys
import time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # to import src from the notebooks folder

import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from src import config, features as ft, metrics, pipelines as pp

pd.set_option("display.max_columns", 50, "display.width", 200)
TARGET, FEATURES = config.TARGET, config.FEATURES
t = pd.read_csv(config.INCIDENT_FILE, parse_dates=["opened_at", "closed_at"])
assert t.shape == (24905, 22)    # table of notebook 03
t = ft.add_time_features(t)
print(t.shape)

(24905, 24)


## 2. Features

I use only information available when the incident is opened, based on notebooks 02 to 04. All features are categorical. Hour and weekday are extracted from `opened_at`.

| Feature | Reason |
|---|---|
| `priority`, `impact`, `urgency` | Available at opening, although they contain similar information |
| `category`, `subcategory`, `u_symptom` | Show differences in resolution time |
| `assignment_group` | First assigned group, with clear differences in resolution time |
| `location`, `opened_by`, `contact_type` | Available at opening; rare values are grouped in the pipeline |
| `sys_created_by` | Strong relationship with the target, but will also be tested without it |
| Hour, weekday | Capture differences related to opening time |

**Left out:** `caller_id` (too many rare values), `notify` (only 36 incidents), `cmdb_ci` (99.7% empty), month (most data comes from February to May), and all columns recorded after opening.

In [7]:
# no feature can be the target, the key, or an analysis-only column
assert not set(FEATURES) & set(config.ANALYSIS_ONLY_COLS + [TARGET, config.ID_COL])
print(len(FEATURES), "features")

13 features


## 3. Split by date

Since the model will predict future incidents, I split the data by `opened_at`: the oldest 60% for training, the next 20% for validation and the last 20% for testing. The split is done by incident, not by event. I also check the three sets and weekly distributions because notebook 04 showed changes between March and April.

In [8]:
t = ft.split_by_date(t)

stats = {"median": (TARGET, "median"), "mean": (TARGET, "mean"), "p90": (TARGET, lambda s: s.quantile(0.9))}
split_summary = t.groupby("split").agg(
    n=("opened_at", "size"), first_opened=("opened_at", "min"), last_opened=("opened_at", "max"), **stats,
    pct_created_resolved=("first_event_state", lambda s: (s == "Resolved").mean() * 100),
).reindex(["train", "validation", "test"])
split_summary[["median", "mean", "p90", "pct_created_resolved"]] = split_summary[["median", "mean", "p90", "pct_created_resolved"]].round(1)

# the three parts follow each other in time
d = t.groupby("split")["opened_at"].agg(["min", "max"])
assert d.loc["train", "max"] < d.loc["validation", "min"] and d.loc["validation", "max"] < d.loc["test", "min"]

config.TABLES_DIR.mkdir(parents=True, exist_ok=True)
split_summary.to_csv(config.TABLES_DIR / "split_summary.csv")
split_summary

,n,first_opened,last_opened,median,mean,p90,pct_created_resolved
split,,,,,,,
train,14943,2016-02-29 01:16:00,2016-04-20 19:57:00,26.6,196.4,406.2,8.8
validation,4981,2016-04-20 19:59:00,2016-05-10 22:27:00,1.3,117.0,250.1,6.7
test,4981,2016-05-10 22:41:00,2017-02-16 14:17:00,2.6,130.2,311.1,3.4


The split follows time order, with about 15,000 incidents in train and 5,000 in each of validation and test. However, their median resolution times differ: 26.6 h, 1.3 h and 2.6 h.

The share of incidents created already `Resolved` also decreases from 8.8% to 3.4%. Most test incidents are from May, with only 264 (5.3%) opened after May.

In [9]:
# week by week, to see where the distribution changes
weekly = t.groupby(t["opened_at"].dt.to_period("W"))[TARGET].agg(
    n="size", median="median", p90=lambda s: s.quantile(0.9)).round(1)
weekly[weekly["n"] >= 100]

,n,median,p90
opened_at,,,
2016-02-29/2016-03-06,1858,76.4,647.2
2016-03-07/2016-03-13,2071,91.0,525.9
2016-03-14/2016-03-20,1975,49.6,369.2
2016-03-21/2016-03-27,1686,26.6,367.6
2016-03-28/2016-04-03,2174,6.5,291.7
2016-04-04/2016-04-10,2008,8.0,331.3
2016-04-11/2016-04-17,1994,2.5,290.8
2016-04-18/2016-04-24,1401,2.9,358.0
2016-04-25/2016-05-01,2046,0.9,170.0


The main change happens in March. The median drops from 76–91 h at the beginning of the month to 6.5 h in the week of 28 March. After that, it stays between 0.9 and 8.0 h until the end of May. The data therefore shows two different periods, but the reason is unknown.

Validation and test belong to the second period and have similar distributions. Train contains both periods.

**Decision:** I keep the date-based split because the goal is to predict future incidents. In the next notebook, I will also test training only on data from 28 March onwards.

## 4. Preprocessing and models

All models use the same preprocessing pipeline (`src/pipelines.py`): one-hot encoding, missing values treated as a separate category, and categories with fewer than 30 training incidents grouped as rare. Categories not seen during training are also grouped there. The pipeline is fitted on the training set only.

**Models:**
- **Baseline:** Predicts the training median.
- **Ridge:** Linear model with regularisation.
- **Random Forest:** 200 trees, minimum 5 incidents per leaf.
- **HistGradientBoosting:** Default settings.

I use MAE, RMSE and RMSLE to compare the models. Negative predictions are set to 0.

In [10]:
train = t[t["split"] == "train"]
val = t[t["split"] == "validation"]
RS = config.RANDOM_STATE

models = {
    "Baseline (median)": DummyRegressor(strategy="median"),
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=RS),
    "HistGradientBoosting": HistGradientBoostingRegressor(random_state=RS),
}

rows = []
for name, model in models.items():
    pipe = Pipeline([("prep", pp.make_preprocessor()), ("model", model)])
    start = time.time()
    pipe.fit(train[FEATURES], train[TARGET])
    seconds = time.time() - start
    on_val = metrics.regression_metrics(val[TARGET], pipe.predict(val[FEATURES]))
    on_train = metrics.regression_metrics(train[TARGET], pipe.predict(train[FEATURES]))
    rows.append({"model": name, "MAE": on_val["MAE"], "RMSE": on_val["RMSE"], "RMSLE": on_val["RMSLE"],
                 "MAE train": on_train["MAE"], "fit seconds": seconds})

results = pd.DataFrame(rows).set_index("model").round(2)
results.to_csv(config.TABLES_DIR / "model_comparison_validation.csv")
results

,MAE,RMSE,RMSLE,MAE train,fit seconds
model,,,,,
Baseline (median),122.46,433.94,2.59,192.04,0.22
Ridge,162.58,385.05,2.98,192.64,0.21
Random Forest,149.19,382.92,2.72,151.32,28.41
HistGradientBoosting,153.41,380.24,3.01,174.58,2.40


On the validation set, all three models improve RMSE compared to the baseline (433.9 h), with scores between 380.2 and 385.1 h. However, none improves MAE or RMSLE.

The gain is therefore limited. The difference between train and validation medians (26.6 h vs 1.3 h) may explain part of the results. The long incidents may also affect predictions because the models use squared error.

These are possible explanations that will be tested in the next notebook.

Random Forest shows no clear overfitting on MAE (151.3 h on train, 149.2 h on validation), but it is slower than HistGradientBoosting (66 s vs 9 s).

## Decisions

- **Features:** The 13 features listed above, all available at opening. I exclude `caller_id`, `notify`, `cmdb_ci`, the month, and all columns created after opening.
- **Split:** Incidents are split by opening date: 60% for training, 20% for validation, and 20% for testing. I keep the test set untouched until the end. Validation and test belong to the same period, while the first four weeks of training are different.
- **Preprocessing:** All models use the same pipeline, fitted on the training set only.
- **Main metric:** I use **MAE on the validation set** to choose the best model. It is measured in hours and is less affected by a few incidents with very long resolution times than RMSE. I also report RMSE and RMSLE. This choice is made before tuning.